In [1]:
import pyspark
print(pyspark.__version__)

4.3.0


In [2]:
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .appName("Sales ETL Project")
    .master("local[*]")
    .getOrCreate()
)
print(spark.version)


C:\Users\peter\Desktop\pyspark-sales-etl\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


4.3.0


In [3]:
data = [
    (1, "Laptop", 3500),
    (2, "Mouse", 120),
    (3, "Keyboard", 250),
    (4, "Monitor", 1200)
]
columns = ["product_id","product_name","price"]
df = spark.createDataFrame(data, columns)
df.show()

+----------+------------+-----+
|product_id|product_name|price|
+----------+------------+-----+
|         1|      Laptop| 3500|
|         2|       Mouse|  120|
|         3|    Keyboard|  250|
|         4|     Monitor| 1200|
+----------+------------+-----+



In [4]:
df.printSchema()

root
 |-- product_id: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- price: long (nullable = true)



In [5]:
df.count()

4

In [6]:
df.select("product_name", "price").show()

+------------+-----+
|product_name|price|
+------------+-----+
|      Laptop| 3500|
|       Mouse|  120|
|    Keyboard|  250|
|     Monitor| 1200|
+------------+-----+



In [7]:
df.filter(df.price > 1000).show()

+----------+------------+-----+
|product_id|product_name|price|
+----------+------------+-----+
|         1|      Laptop| 3500|
|         4|     Monitor| 1200|
+----------+------------+-----+



In [8]:
df.filter(df.product_name == "Laptop").show()

+----------+------------+-----+
|product_id|product_name|price|
+----------+------------+-----+
|         1|      Laptop| 3500|
+----------+------------+-----+



In [9]:
df.filter(
    (df.price > 100) & (df.price < 2000)
).show()

+----------+------------+-----+
|product_id|product_name|price|
+----------+------------+-----+
|         2|       Mouse|  120|
|         3|    Keyboard|  250|
|         4|     Monitor| 1200|
+----------+------------+-----+



In [10]:
from pyspark.sql.functions import col
df = df.withColumn(
    "price_with_vat",
    col("price")*1.23
)
df.show()

+----------+------------+-----+--------------+
|product_id|product_name|price|price_with_vat|
+----------+------------+-----+--------------+
|         1|      Laptop| 3500|        4305.0|
|         2|       Mouse|  120|         147.6|
|         3|    Keyboard|  250|         307.5|
|         4|     Monitor| 1200|        1476.0|
+----------+------------+-----+--------------+



In [11]:
df.select("product_name","price").show()
df.filter(df.price > 200).show()
df = df.withColumn(
    "price_discount",
    col("price")*0.9
)
df.show()
df.orderBy(col("price").desc()).show()





+------------+-----+
|product_name|price|
+------------+-----+
|      Laptop| 3500|
|       Mouse|  120|
|    Keyboard|  250|
|     Monitor| 1200|
+------------+-----+

+----------+------------+-----+--------------+
|product_id|product_name|price|price_with_vat|
+----------+------------+-----+--------------+
|         1|      Laptop| 3500|        4305.0|
|         3|    Keyboard|  250|         307.5|
|         4|     Monitor| 1200|        1476.0|
+----------+------------+-----+--------------+

+----------+------------+-----+--------------+--------------+
|product_id|product_name|price|price_with_vat|price_discount|
+----------+------------+-----+--------------+--------------+
|         1|      Laptop| 3500|        4305.0|        3150.0|
|         2|       Mouse|  120|         147.6|         108.0|
|         3|    Keyboard|  250|         307.5|         225.0|
|         4|     Monitor| 1200|        1476.0|        1080.0|
+----------+------------+-----+--------------+--------------+

+---

In [12]:
print(type(df))

<class 'pyspark.sql.classic.dataframe.DataFrame'>


In [13]:
from pyspark.sql.functions import sum, avg, max, min, count
df.select(
    sum("price").alias("total_price"),
    avg("price").alias("average_price"),
    max("price").alias("max_price"),
    min("price").alias("min_price"),
    count("product_id").alias("number_of_products")
).show()

+-----------+-------------+---------+---------+------------------+
|total_price|average_price|max_price|min_price|number_of_products|
+-----------+-------------+---------+---------+------------------+
|       5070|       1267.5|     3500|      120|                 4|
+-----------+-------------+---------+---------+------------------+



In [14]:
df.select(
    avg("price").alias("average_price"),
    min("price").alias("min_price"),
    max("price").alias("max_price")


).show()
df.orderBy(col("product_name")).show()

+-------------+---------+---------+
|average_price|min_price|max_price|
+-------------+---------+---------+
|       1267.5|      120|     3500|
+-------------+---------+---------+

+----------+------------+-----+--------------+--------------+
|product_id|product_name|price|price_with_vat|price_discount|
+----------+------------+-----+--------------+--------------+
|         3|    Keyboard|  250|         307.5|         225.0|
|         1|      Laptop| 3500|        4305.0|        3150.0|
|         4|     Monitor| 1200|        1476.0|        1080.0|
|         2|       Mouse|  120|         147.6|         108.0|
+----------+------------+-----+--------------+--------------+



In [15]:
df_orders=spark.read.csv("C:/Users/peter/Desktop/pyspark-sales-etl/data/raw/orders.csv", header=True, inferSchema=True)
df_orders.show()

+--------+----------+--------+----------+----------+-------+
|order_id|product_id|quantity|unit_price|order_date|country|
+--------+----------+--------+----------+----------+-------+
|       1|       101|       2|      3500|2026-01-10| Poland|
|       2|       102|       5|       120|2026-01-11|Germany|
|       3|       103|       3|       250|2026-01-12| Poland|
|       4|       104|       1|      1200|2026-01-13| France|
|       5|       101|       1|      3500|2026-01-14|Germany|
|       6|       105|       4|        80|2026-01-15| Poland|
|       7|       102|       2|       120|2026-01-16| France|
|       8|       103|       1|       250|2026-01-17|Germany|
+--------+----------+--------+----------+----------+-------+



In [16]:
df_orders.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- country: string (nullable = true)



In [17]:
df_orders.count()

8

In [18]:
df_orders.show(10,truncate=False)

+--------+----------+--------+----------+----------+-------+
|order_id|product_id|quantity|unit_price|order_date|country|
+--------+----------+--------+----------+----------+-------+
|1       |101       |2       |3500      |2026-01-10|Poland |
|2       |102       |5       |120       |2026-01-11|Germany|
|3       |103       |3       |250       |2026-01-12|Poland |
|4       |104       |1       |1200      |2026-01-13|France |
|5       |101       |1       |3500      |2026-01-14|Germany|
|6       |105       |4       |80        |2026-01-15|Poland |
|7       |102       |2       |120       |2026-01-16|France |
|8       |103       |1       |250       |2026-01-17|Germany|
+--------+----------+--------+----------+----------+-------+



In [19]:
df_orders.show(5)

+--------+----------+--------+----------+----------+-------+
|order_id|product_id|quantity|unit_price|order_date|country|
+--------+----------+--------+----------+----------+-------+
|       1|       101|       2|      3500|2026-01-10| Poland|
|       2|       102|       5|       120|2026-01-11|Germany|
|       3|       103|       3|       250|2026-01-12| Poland|
|       4|       104|       1|      1200|2026-01-13| France|
|       5|       101|       1|      3500|2026-01-14|Germany|
+--------+----------+--------+----------+----------+-------+
only showing top 5 rows


In [20]:
df_orders.count()

8

In [21]:
from pyspark.sql.functions import col, sum
df_orders.select(
    *[
        sum(col(c).isNull().cast("int")).alias(c)
        for c in df_orders.columns
    ]
).show()

+--------+----------+--------+----------+----------+-------+
|order_id|product_id|quantity|unit_price|order_date|country|
+--------+----------+--------+----------+----------+-------+
|       0|         0|       0|         0|         0|      0|
+--------+----------+--------+----------+----------+-------+



In [22]:
df_orders.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- country: string (nullable = true)



In [23]:
df_orders.show(5)

+--------+----------+--------+----------+----------+-------+
|order_id|product_id|quantity|unit_price|order_date|country|
+--------+----------+--------+----------+----------+-------+
|       1|       101|       2|      3500|2026-01-10| Poland|
|       2|       102|       5|       120|2026-01-11|Germany|
|       3|       103|       3|       250|2026-01-12| Poland|
|       4|       104|       1|      1200|2026-01-13| France|
|       5|       101|       1|      3500|2026-01-14|Germany|
+--------+----------+--------+----------+----------+-------+
only showing top 5 rows


In [24]:
df_orders.count()

8

In [25]:
from pyspark.sql.functions import col, sum

df_orders.select(
    *[
        sum(col(c).isNull().cast("int")).alias(c)
        for c in df_orders.columns
    ]
).show()

+--------+----------+--------+----------+----------+-------+
|order_id|product_id|quantity|unit_price|order_date|country|
+--------+----------+--------+----------+----------+-------+
|       0|         0|       0|         0|         0|      0|
+--------+----------+--------+----------+----------+-------+



In [26]:
from pyspark.sql.functions import col
df_orders=df_orders.withColumn(
    "total_amount",
    col("quantity")*col("unit_price")
)
df_orders.show()

+--------+----------+--------+----------+----------+-------+------------+
|order_id|product_id|quantity|unit_price|order_date|country|total_amount|
+--------+----------+--------+----------+----------+-------+------------+
|       1|       101|       2|      3500|2026-01-10| Poland|        7000|
|       2|       102|       5|       120|2026-01-11|Germany|         600|
|       3|       103|       3|       250|2026-01-12| Poland|         750|
|       4|       104|       1|      1200|2026-01-13| France|        1200|
|       5|       101|       1|      3500|2026-01-14|Germany|        3500|
|       6|       105|       4|        80|2026-01-15| Poland|         320|
|       7|       102|       2|       120|2026-01-16| France|         240|
|       8|       103|       1|       250|2026-01-17|Germany|         250|
+--------+----------+--------+----------+----------+-------+------------+



In [27]:
from pyspark.sql.functions import sum
df_country_sales = df_orders.groupBy("country").agg(
    sum("total_amount").alias("total_revenue")
)
df_country_sales.show()

+-------+-------------+
|country|total_revenue|
+-------+-------------+
|Germany|         4350|
| France|         1440|
| Poland|         8070|
+-------+-------------+



In [28]:
df_country_sales.write \
    .mode("overwrite") \
    .parquet("C:/Users/peter/Desktop/pyspark-sales-etl/data/processed/sales_by_country")

In [29]:
print(spark.version)

4.3.0


In [30]:
print(
    spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion()
)

3.5.0


In [31]:
print(spark.sparkContext.getConf().get("spark.hadoop.fs.permissions.enabled", "not set"))

not set


In [32]:
import os

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] += os.pathsep + r"C:\hadoop\bin"

print(os.environ["HADOOP_HOME"])

C:\hadoop


In [33]:
spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion()

'3.5.0'

In [34]:
import pyspark
print("PySpark:", pyspark.__version__)

print("Hadoop:", spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion())

PySpark: 4.3.0
Hadoop: 3.5.0


In [35]:
import os

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] += os.pathsep + r"C:\hadoop\bin"

print(os.path.exists(
    r"C:\hadoop\bin\winutils.exe"
))

True


In [36]:
df_country_sales.write \
    .mode("overwrite") \
    .parquet(
        "C:/Users/peter/Desktop/pyspark-sales-etl/data/processed/sales_by_country"
    )

In [37]:
import os
import shutil

print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))
print("winutils:", shutil.which("winutils.exe"))
print(
    "Plik istnieje:",
    os.path.exists(r"C:\hadoop\bin\winutils.exe")
)

HADOOP_HOME: C:\hadoop
winutils: C:\hadoop\bin\winutils.exe
Plik istnieje: True


In [38]:
df_country_sales.write \
    .mode("overwrite") \
    .parquet(
        "C:/Users/peter/Desktop/pyspark-sales-etl/data/processed/sales_by_country"
    )

In [39]:
import os

path = "data/processed/sales_by_country"

print(os.listdir(path))

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'data/processed/sales_by_country'

In [40]:
import os

print(os.getcwd())

C:\Users\peter\Desktop\pyspark-sales-etl\notebooks


In [41]:
path = r"C:\Users\peter\Desktop\pyspark-sales-etl\data\processed\sales_by_country"

print(os.path.exists(path))

True


In [42]:
print(os.listdir(path))

['.part-00000-20d988e7-9b79-4704-ae53-1c049f3f8da2-c000.snappy.parquet.crc', '._SUCCESS.crc', 'part-00000-20d988e7-9b79-4704-ae53-1c049f3f8da2-c000.snappy.parquet', '_SUCCESS']


In [43]:
df_check = spark.read.parquet(output_path)

df_check.show()

print("Liczba rekordów:", df_check.count())

NameError: name 'output_path' is not defined

In [45]:
import os

# Ścieżka do zapisanych danych
output_path = r"C:\Users\peter\Desktop\pyspark-sales-etl\data\processed\sales_by_country"

# Odczyt danych Parquet
df_check = spark.read.parquet(output_path)

# Wyświetlenie danych
df_check.show()

# Liczba rekordów
print("Liczba rekordów:", df_check.count())

+-------+-------------+
|country|total_revenue|
+-------+-------------+
|Germany|         4350|
| France|         1440|
| Poland|         8070|
+-------+-------------+

Liczba rekordów: 3
